In [5]:
import sys, os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from qiskit.quantum_info import SparsePauliOp
from qiskit.circuit.library import efficient_su2

from src.runners import run_vqe_trace
from src.transformations import (
    vqe_identity,
    vqe_barrier,
    vqe_identity_xx,
    vqe_global_phase,
    vqe_fault_x,
    vqe_fault_replace_rotation,
    vqe_fault_shift_parameter,
    vqe_fault_change_entanglement,
    vqe_fault_shift_parameter_strong,
    vqe_fault_replace_rotation_strong,

)
from src.checker import evaluate_vqe_pair, compare_vqe_traces

In [6]:
sns.set_style("whitegrid")
plt.rcParams.update({"font.size": 12})

project_root = Path.cwd().parent
results_dir = project_root / "results"
figures_dir = results_dir / "figures"

results_dir.mkdir(parents=True, exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)

print("VQE Metamorphic Testing Notebook")
print("=" * 50)
print(f"Project root: {project_root}")
print(f"Results dir: {results_dir}")
print(f"Figures dir: {figures_dir}")

VQE Metamorphic Testing Notebook
Project root: C:\Users\shask\Desktop\Winter 2026\CSI 5370\Project paper\metamorphic_quantum_testing
Results dir: C:\Users\shask\Desktop\Winter 2026\CSI 5370\Project paper\metamorphic_quantum_testing\results
Figures dir: C:\Users\shask\Desktop\Winter 2026\CSI 5370\Project paper\metamorphic_quantum_testing\results\figures


In [7]:
# CONFIGURATION

NUM_QUBITS = 4
NUM_RUNS = 30

# Original/full oracle thresholds
ENERGY_THRESHOLD = 1e-3
SYMMETRY_THRESHOLD = 1e-3
PARAM_THRESHOLD = 1e-2

REPS = 5
MAXITER = 120
RANDOM_SEED = 42

# Ablation oracle thresholds
ENERGY_ONLY_THRESHOLD = 1e-3
TRACE_METRIC_NAME = "avg_energy_diff"
TRACE_THRESHOLD = 1e-2

# ============================================================
# HAMILTONIAN AND SYMMETRY OBSERVABLE
# ============================================================

H = SparsePauliOp(
    ["ZZII", "IIZZ", "XXII", "IIXX"],
    coeffs=[-1.0, -1.0, 0.5, 0.5]
)

symmetry_op = SparsePauliOp(
    ["ZZII", "IIZZ"],
    coeffs=[1.0, 1.0]
)

# ============================================================
# COMMUTATOR CHECK
# ============================================================

def check_commutation(A, B, label_A="A", label_B="B"):
    """
    Check whether two SparsePauliOp operators commute.
    """
    comm = (A @ B - B @ A).simplify(atol=1e-12)

    print(f"\nCommutator [{label_A}, {label_B}]:")
    print(comm)

    if len(comm.paulis) == 0 or np.allclose(comm.coeffs, 0.0):
        print("Result: operators commute.\n")
        return True
    else:
        print("Result: operators do NOT commute.\n")
        return False


commutes = check_commutation(H, symmetry_op, "H", "symmetry_op")

# ============================================================
# ANSATZ
# ============================================================

ansatz = efficient_su2(
    NUM_QUBITS,
    reps=REPS,
    entanglement="full"
)

print("Original ansatz:")
print(f"Number of qubits: {ansatz.num_qubits}")
print(f"Number of parameters: {ansatz.num_parameters}")
print(f"Circuit depth: {ansatz.depth()}")

# ============================================================
# METAMORPHIC RELATIONS
# ============================================================

relations = {
    "identity": {
        "transform": vqe_identity,
        "type": "valid",
    },
    "barrier": {
        "transform": vqe_barrier,
        "type": "valid",
    },
    "identity_xx": {
        "transform": vqe_identity_xx,
        "type": "valid",
    },
    "fault_x": {
        "transform": vqe_fault_x,
        "type": "fault",
    },
    "fault_replace_rot": {
        "transform": vqe_fault_replace_rotation,
        "type": "fault",
    },
    "fault_shift_param": {
        "transform": vqe_fault_shift_parameter,
        "type": "fault",
    },
    "fault_change_entanglement": {
        "transform": vqe_fault_change_entanglement,
        "type": "fault",
    },
    "fault_replace_rot_strong": {
        "transform": vqe_fault_replace_rotation_strong,
        "type": "fault",
    },
    "fault_shift_param_strong": {
        "transform": vqe_fault_shift_parameter_strong,
        "type": "fault",
    },
}

# ============================================================
# HELPERS
# ============================================================

def apply_transform(transform, ansatz, seed):
    """
    Apply a VQE transformation.
    Some transformations accept seed, while some valid ones do not.
    """
    try:
        return transform(ansatz, seed=seed)
    except TypeError:
        return transform(ansatz)


def evaluate_vqe_pair_energy_only(
    source_result,
    follow_result,
    energy_threshold: float = 1e-3,
):
    """
    Baseline oracle:
    only checks final energy difference.
    """
    delta_E = abs(float(source_result.fun) - float(follow_result.fun))
    violation = delta_E > energy_threshold

    return {
        "violation": bool(violation),
        "delta_E": float(delta_E),
        "energy_violation": bool(violation),
    }


def evaluate_vqe_pair_energy_trace(
    source_result,
    follow_result,
    source_trace: pd.DataFrame,
    follow_trace: pd.DataFrame,
    energy_threshold: float = 1e-3,
    trace_threshold: float = 1e-2,
    trace_metric_name: str = "avg_energy_diff",
):
    """
    Gray-box oracle:
    checks final energy difference and one trajectory metric.
    """
    trace_metrics = compare_vqe_traces(source_trace, follow_trace)

    delta_E = abs(float(source_result.fun) - float(follow_result.fun))
    trace_value = float(trace_metrics[trace_metric_name])

    energy_violation = delta_E > energy_threshold
    trace_violation = trace_value > trace_threshold
    violation = energy_violation or trace_violation

    return {
        "violation": bool(violation),
        "delta_E": float(delta_E),
        "energy_violation": bool(energy_violation),
        "trace_violation": bool(trace_violation),
        "trace_metric_name": trace_metric_name,
        "trace_value": trace_value,
        **trace_metrics,
    }


def summarize_binary_performance(df):
    """
    Expects:
      - df['type'] in {'valid', 'fault'}
      - df['violation'] as bool
    """
    tp = int(((df["type"] == "fault") & (df["violation"])).sum())
    fn = int(((df["type"] == "fault") & (~df["violation"])).sum())
    fp = int(((df["type"] == "valid") & (df["violation"])).sum())
    tn = int(((df["type"] == "valid") & (~df["violation"])).sum())

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) else 0.0
    )

    false_positive_rate = 100 * fp / (fp + tn) if (fp + tn) else 0.0
    true_positive_rate = 100 * tp / (tp + fn) if (tp + fn) else 0.0

    return {
        "false_positive_rate": round(false_positive_rate, 2),
        "true_positive_rate": round(true_positive_rate, 2),
        "precision": round(precision, 4),
        "recall": round(recall, 4),
        "f1_score": round(f1, 4),
        "tp": tp,
        "fp": fp,
        "tn": tn,
        "fn": fn,
    }


def summarize_by_oracle(df_runs):
    rows = []

    for oracle_name, g in df_runs.groupby("oracle_variant"):
        stats = summarize_binary_performance(g)
        rows.append({
            "oracle_variant": oracle_name,
            **stats,
        })

    return pd.DataFrame(rows).sort_values(
        ["false_positive_rate", "f1_score", "true_positive_rate"],
        ascending=[True, False, False]
    ).reset_index(drop=True)


def summarize_by_oracle_and_relation(df_runs):
    rows = []

    for (oracle_name, relation_name, relation_type), g in df_runs.groupby(
        ["oracle_variant", "relation", "type"]
    ):
        row = {
            "oracle_variant": oracle_name,
            "relation": relation_name,
            "type": relation_type,
            "num_runs": len(g),
            "detection_rate": round(100 * g["violation"].mean(), 2),
            "mean_delta_E": round(g["delta_E"].mean(), 6),
        }

        if "trace_value" in g.columns and g["trace_value"].notna().any():
            row["mean_trace_value"] = round(g["trace_value"].mean(), 6)

        rows.append(row)

    return pd.DataFrame(rows).sort_values(
        ["oracle_variant", "type", "relation"]
    ).reset_index(drop=True)


# ============================================================
# PRE-GENERATE INITIAL POINTS
# ============================================================

rng = np.random.default_rng(RANDOM_SEED)

initial_points = [
    rng.uniform(-0.1, 0.1, ansatz.num_parameters)
    for _ in range(NUM_RUNS)
]

# ============================================================
# MAIN EXPERIMENT
# ============================================================

results = []
run_level_results = []

print(f"\nRunning VQE metamorphic testing ablation ({NUM_RUNS} paired runs per relation)...\n")

for relation_name, relation_info in relations.items():

    transform = relation_info["transform"]
    relation_type = relation_info["type"]

    print(f"→ Testing: {relation_name}")

    # Original/full oracle summary lists
    delta_E_list = []
    avg_energy_diff_list = []
    max_energy_diff_list = []

    sym_diff_list = []
    max_sym_diff_list = []

    param_diff_list = []
    max_param_diff_list = []

    detected_list = []

    for run in range(NUM_RUNS):

        initial_point = initial_points[run]

        # ----------------------------------------------------
        # SOURCE RUN
        # ----------------------------------------------------
        source_result, source_trace = run_vqe_trace(
            ansatz=ansatz,
            hamiltonian=H,
            symmetry_op=symmetry_op,
            initial_point=initial_point,
            maxiter=MAXITER,
        )

        # ----------------------------------------------------
        # FOLLOW-UP ANSATZ
        # ----------------------------------------------------
        follow_ansatz = apply_transform(
            transform=transform,
            ansatz=ansatz,
            seed=1000 + run,
        )

        if follow_ansatz.num_parameters == ansatz.num_parameters:
            follow_initial_point = initial_point
        else:
            follow_initial_point = rng.uniform(
                -0.1,
                0.1,
                follow_ansatz.num_parameters,
            )

        # ----------------------------------------------------
        # FOLLOW-UP RUN
        # ----------------------------------------------------
        follow_result, follow_trace = run_vqe_trace(
            ansatz=follow_ansatz,
            hamiltonian=H,
            symmetry_op=symmetry_op,
            initial_point=follow_initial_point,
            maxiter=MAXITER,
        )

        # ----------------------------------------------------
        # ORIGINAL / FULL ORACLE
        # ----------------------------------------------------
        full_metrics = evaluate_vqe_pair(
            source_result,
            follow_result,
            source_trace,
            follow_trace,
            energy_threshold=ENERGY_THRESHOLD,
            symmetry_threshold=SYMMETRY_THRESHOLD,
            param_threshold=PARAM_THRESHOLD,
        )

        delta_E_list.append(full_metrics["delta_E"])
        avg_energy_diff_list.append(full_metrics["avg_energy_diff"])
        max_energy_diff_list.append(full_metrics["max_energy_diff"])

        sym_diff_list.append(full_metrics["avg_sym_diff"])
        max_sym_diff_list.append(full_metrics["max_sym_diff"])

        param_diff_list.append(full_metrics["avg_param_diff"])
        max_param_diff_list.append(full_metrics["max_param_diff"])

        detected_list.append(full_metrics["violation"])

        run_level_results.append({
            "oracle_variant": "full_original_oracle",
            "relation": relation_name,
            "type": relation_type,
            "run": run + 1,
            "source_num_parameters": ansatz.num_parameters,
            "follow_num_parameters": follow_ansatz.num_parameters,
            "source_depth": ansatz.depth(),
            "follow_depth": follow_ansatz.depth(),
            "delta_E": round(full_metrics["delta_E"], 8),
            "avg_energy_diff": round(full_metrics["avg_energy_diff"], 8),
            "max_energy_diff": round(full_metrics["max_energy_diff"], 8),
            "avg_sym_diff": round(full_metrics["avg_sym_diff"], 8),
            "max_sym_diff": round(full_metrics["max_sym_diff"], 8),
            "avg_param_diff": round(full_metrics["avg_param_diff"], 8),
            "max_param_diff": round(full_metrics["max_param_diff"], 8),
            "trace_len": full_metrics["trace_len"],
            "violation": full_metrics["violation"],
            "trace_value": np.nan,
        })

        # ----------------------------------------------------
        # ENERGY-ONLY ORACLE
        # ----------------------------------------------------
        energy_only_metrics = evaluate_vqe_pair_energy_only(
            source_result,
            follow_result,
            energy_threshold=ENERGY_ONLY_THRESHOLD,
        )

        run_level_results.append({
            "oracle_variant": "energy_only",
            "relation": relation_name,
            "type": relation_type,
            "run": run + 1,
            "source_num_parameters": ansatz.num_parameters,
            "follow_num_parameters": follow_ansatz.num_parameters,
            "source_depth": ansatz.depth(),
            "follow_depth": follow_ansatz.depth(),
            "delta_E": round(energy_only_metrics["delta_E"], 8),
            "avg_energy_diff": np.nan,
            "max_energy_diff": np.nan,
            "avg_sym_diff": np.nan,
            "max_sym_diff": np.nan,
            "avg_param_diff": np.nan,
            "max_param_diff": np.nan,
            "trace_len": np.nan,
            "violation": energy_only_metrics["violation"],
            "trace_value": np.nan,
        })

        # ----------------------------------------------------
        # ENERGY + TRACE ORACLE
        # ----------------------------------------------------
        energy_trace_metrics = evaluate_vqe_pair_energy_trace(
            source_result,
            follow_result,
            source_trace,
            follow_trace,
            energy_threshold=ENERGY_ONLY_THRESHOLD,
            trace_threshold=TRACE_THRESHOLD,
            trace_metric_name=TRACE_METRIC_NAME,
        )

        run_level_results.append({
            "oracle_variant": "energy_plus_trace",
            "relation": relation_name,
            "type": relation_type,
            "run": run + 1,
            "source_num_parameters": ansatz.num_parameters,
            "follow_num_parameters": follow_ansatz.num_parameters,
            "source_depth": ansatz.depth(),
            "follow_depth": follow_ansatz.depth(),
            "delta_E": round(energy_trace_metrics["delta_E"], 8),
            "avg_energy_diff": round(energy_trace_metrics["avg_energy_diff"], 8),
            "max_energy_diff": round(energy_trace_metrics["max_energy_diff"], 8),
            "avg_sym_diff": round(energy_trace_metrics["avg_sym_diff"], 8),
            "max_sym_diff": round(energy_trace_metrics["max_sym_diff"], 8),
            "avg_param_diff": round(energy_trace_metrics["avg_param_diff"], 8),
            "max_param_diff": round(energy_trace_metrics["max_param_diff"], 8),
            "trace_len": energy_trace_metrics["trace_len"],
            "violation": energy_trace_metrics["violation"],
            "trace_value": round(energy_trace_metrics["trace_value"], 8),
        })

    # --------------------------------------------------------
    # ORIGINAL SUMMARY TABLE
    # --------------------------------------------------------
    results.append({
        "relation": relation_name,
        "type": relation_type,
        "mean_delta_E": round(np.mean(delta_E_list), 6),
        "std_delta_E": round(np.std(delta_E_list), 6),
        "mean_avg_energy_diff": round(np.mean(avg_energy_diff_list), 6),
        "mean_max_energy_diff": round(np.mean(max_energy_diff_list), 6),
        "mean_sym_diff": round(np.mean(sym_diff_list), 6),
        "mean_max_sym_diff": round(np.mean(max_sym_diff_list), 6),
        "mean_param_diff": round(np.mean(param_diff_list), 6),
        "mean_max_param_diff": round(np.mean(max_param_diff_list), 6),
        "detection_rate": round(np.mean(detected_list) * 100, 2),
    })

# ============================================================
# SAVE RESULTS
# ============================================================

df_vqe = pd.DataFrame(results)
df_vqe_runs = pd.DataFrame(run_level_results)

df_vqe.to_csv(results_dir / "vqe_results_4q.csv", index=False)
df_vqe_runs.to_csv(
    results_dir / "vqe_run_level_results_4q_with_oracle_ablation.csv",
    index=False
)

print("\nExperiment completed.")
print(f"Summary results saved to: {results_dir / 'vqe_results_4q.csv'}")
print(
    "Run-level results with oracle ablation saved to: "
    f"{results_dir / 'vqe_run_level_results_4q_with_oracle_ablation.csv'}"
)

display(df_vqe)
display(df_vqe_runs.head())

# ============================================================
# ORACLE COMPARISON SUMMARIES
# ============================================================

df_oracle_summary = summarize_by_oracle(df_vqe_runs)
df_oracle_relation_summary = summarize_by_oracle_and_relation(df_vqe_runs)

print("\nOracle-level summary:")
display(df_oracle_summary)

print("\nOracle-by-relation summary:")
display(df_oracle_relation_summary)

# Optional: compare only the two main baselines
df_compare = df_vqe_runs[
    df_vqe_runs["oracle_variant"].isin(["energy_only", "energy_plus_trace"])
].copy()

print("\nBaseline comparison only:")
display(summarize_by_oracle(df_compare))

print("\nBaseline comparison by relation:")
display(summarize_by_oracle_and_relation(df_compare))


Commutator [H, symmetry_op]:
SparsePauliOp(['IIII'],
              coeffs=[0.+0.j])
Result: operators commute.

Original ansatz:
Number of qubits: 4
Number of parameters: 48
Circuit depth: 33

Running VQE metamorphic testing ablation (30 paired runs per relation)...

→ Testing: identity
→ Testing: barrier
→ Testing: identity_xx
→ Testing: fault_x
→ Testing: fault_replace_rot
→ Testing: fault_shift_param
→ Testing: fault_change_entanglement
→ Testing: fault_replace_rot_strong
→ Testing: fault_shift_param_strong

Experiment completed.
Summary results saved to: C:\Users\shask\Desktop\Winter 2026\CSI 5370\Project paper\metamorphic_quantum_testing\results\vqe_results_4q.csv
Run-level results with oracle ablation saved to: C:\Users\shask\Desktop\Winter 2026\CSI 5370\Project paper\metamorphic_quantum_testing\results\vqe_run_level_results_4q_with_oracle_ablation.csv


,relation,type,mean_delta_E,std_delta_E,mean_avg_energy_diff,mean_max_energy_diff,mean_sym_diff,mean_max_sym_diff,mean_param_diff,mean_max_param_diff,detection_rate
0,identity,valid,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0
1,barrier,valid,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0
2,identity_xx,valid,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0
3,fault_x,fault,0.228410,0.216641,0.554910,2.009532,0.395848,1.711397,3.067780,4.175201,100.0
4,fault_replace_rot,fault,0.068360,0.069643,0.105566,0.919561,0.078148,0.845041,1.085081,2.164741,100.0
5,fault_shift_param,fault,0.163269,0.152212,0.245844,1.091033,0.184149,0.970410,1.795452,2.960530,100.0
6,fault_change_entanglement,fault,0.281982,0.190702,0.260649,1.200008,0.161180,1.016901,2.829727,4.072373,100.0
7,fault_replace_rot_strong,fault,0.302711,0.236967,0.572032,1.654105,0.526516,1.548795,3.648947,4.977317,100.0
8,fault_shift_param_strong,fault,0.331737,0.262484,0.636484,1.820952,0.593471,1.787473,3.421922,4.790317,100.0


,oracle_variant,relation,type,run,source_num_parameters,follow_num_parameters,source_depth,follow_depth,delta_E,avg_energy_diff,max_energy_diff,avg_sym_diff,max_sym_diff,avg_param_diff,max_param_diff,trace_len,violation,trace_value
0,full_original_oracle,identity,valid,1,48,48,33,35,0.0,0.0,0.0,0.0,0.0,0.0,0.0,120.0,False,NaN
1,energy_only,identity,valid,1,48,48,33,35,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN
2,energy_plus_trace,identity,valid,1,48,48,33,35,0.0,0.0,0.0,0.0,0.0,0.0,0.0,120.0,False,0.0
3,full_original_oracle,identity,valid,2,48,48,33,35,0.0,0.0,0.0,0.0,0.0,0.0,0.0,120.0,False,NaN
4,energy_only,identity,valid,2,48,48,33,35,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN



Oracle-level summary:


,oracle_variant,false_positive_rate,true_positive_rate,precision,recall,f1_score,tp,fp,tn,fn
0,energy_only,0.0,100.0,1.0,1.0,1.0,180,0,90,0
1,energy_plus_trace,0.0,100.0,1.0,1.0,1.0,180,0,90,0
2,full_original_oracle,0.0,100.0,1.0,1.0,1.0,180,0,90,0


...

,oracle_variant,relation,type,num_runs,detection_rate,mean_delta_E,mean_trace_value
0,energy_only,fault_change_entanglement,fault,30,100.0,0.281982,NaN
1,energy_only,fault_replace_rot,fault,30,100.0,0.068360,NaN
2,energy_only,fault_replace_rot_strong,fault,30,100.0,0.302711,NaN
3,energy_only,fault_shift_param,fault,30,100.0,0.163269,NaN
4,energy_only,fault_shift_param_strong,fault,30,100.0,0.331737,NaN
5,energy_only,fault_x,fault,30,100.0,0.228410,NaN
6,energy_only,barrier,valid,30,0.0,0.000000,NaN
7,energy_only,identity,valid,30,0.0,0.000000,NaN
8,energy_only,identity_xx,valid,30,0.0,0.000000,NaN
9,energy_plus_trace,fault_change_entanglement,fault,30,100.0,0.281982,0.260649



Baseline comparison only:


,oracle_variant,false_positive_rate,true_positive_rate,precision,recall,f1_score,tp,fp,tn,fn
0,energy_only,0.0,100.0,1.0,1.0,1.0,180,0,90,0
1,energy_plus_trace,0.0,100.0,1.0,1.0,1.0,180,0,90,0



Baseline comparison by relation:


,oracle_variant,relation,type,num_runs,detection_rate,mean_delta_E,mean_trace_value
0,energy_only,fault_change_entanglement,fault,30,100.0,0.281982,NaN
1,energy_only,fault_replace_rot,fault,30,100.0,0.068360,NaN
2,energy_only,fault_replace_rot_strong,fault,30,100.0,0.302711,NaN
3,energy_only,fault_shift_param,fault,30,100.0,0.163269,NaN
4,energy_only,fault_shift_param_strong,fault,30,100.0,0.331737,NaN
5,energy_only,fault_x,fault,30,100.0,0.228410,NaN
6,energy_only,barrier,valid,30,0.0,0.000000,NaN
7,energy_only,identity,valid,30,0.0,0.000000,NaN
8,energy_only,identity_xx,valid,30,0.0,0.000000,NaN
9,energy_plus_trace,fault_change_entanglement,fault,30,100.0,0.281982,0.260649
